# Authentication & Data Query Tests

Comprehensive testing for Datacore client:
- Authentication methods
- Dataset registry
- Data queries
- Pagination and pagination helpers
- Error handling
- Performance tips

## Setup & Imports

In [1]:
import os
import sys
from datacore import Datacore
import pandas as pd
from typing import Optional

print("✓ Imports successful")

ModuleNotFoundError: No module named 'requests'

## 1. Authentication Tests

### 1.1 Test: Missing API Key

In [ ]:
# Test: Error when no API key
old_key = os.environ.pop("X_DATACORE_API_KEY", None)

try:
    client = Datacore()
    print("✗ FAIL: Should have raised ValueError")
except ValueError as e:
    print(f"✓ PASS: Correctly raised ValueError")
    print(f"  Error: {str(e)[:60]}...")
finally:
    if old_key:
        os.environ["X_DATACORE_API_KEY"] = old_key

### 1.2 Test: Direct API Key

In [ ]:
# Test: Create client with direct API key
try:
    client = Datacore(api_key="test-key-12345")
    print(f"✓ PASS: Direct key authentication")
    print(f"  API Key: {client.api_key[:10]}...")
    print(f"  Base URL: {client.BASE_URL}")
except Exception as e:
    print(f"✗ FAIL: {e}")

### 1.3 Test: Environment Variable

In [ ]:
# Check if API key exists in environment
api_key = os.getenv("X_DATACORE_API_KEY")

if api_key:
    print(f"✓ API key found in environment")
    print(f"  Key: {api_key[:15]}...")
    
    try:
        client = Datacore()
        print(f"✓ PASS: Env variable authentication")
    except Exception as e:
        print(f"✗ FAIL: {e}")
else:
    print("⚠ No API key in environment")
    print("  To set: $env:X_DATACORE_API_KEY = 'your-key'")

### 1.4 Test: Custom Timeout

In [ ]:
# Test: Custom timeout configuration
try:
    client = Datacore(api_key="test-key", timeout=60)
    print(f"✓ PASS: Custom timeout configuration")
    print(f"  Timeout: {client.timeout}s")
except Exception as e:
    print(f"✗ FAIL: {e}")

## 2. Dataset Registry Tests

### 2.1 Test: List All Datasets

In [ ]:
# Test: List all available datasets
try:
    client = Datacore(api_key="test-key")
    datasets = client.list_datasets()
    
    print(f"✓ PASS: List datasets")
    print(f"  Found {len(datasets)} datasets:")
    print()
    
    for key, info in datasets.items():
        print(f"    • {key:<20} → {info['method_name']:<30}")
        print(f"      {info['description']}")
        print()
except Exception as e:
    print(f"✗ FAIL: {e}")

### 2.2 Test: Get Dataset Info

In [ ]:
# Test: Get specific dataset information
try:
    client = Datacore(api_key="test-key")
    info = client.get_dataset_info("historical_price")
    
    if info:
        print(f"✓ PASS: Get dataset info")
        print(f"  Dataset: historical_price")
        print(f"  Code: {info['code']}")
        print(f"  Method: {info['method_name']}")
        print(f"  Description: {info['description']}")
    else:
        print(f"✗ FAIL: Dataset not found")
except Exception as e:
    print(f"✗ FAIL: {e}")

### 2.3 Test: Invalid Dataset Key

In [ ]:
# Test: Invalid dataset key handling
try:
    client = Datacore(api_key="test-key")
    info = client.get_dataset_info("nonexistent_dataset_xyz")
    
    if info is None:
        print(f"✓ PASS: Invalid dataset handling")
        print(f"  Correctly returned None for invalid key")
    else:
        print(f"✗ FAIL: Should return None")
except Exception as e:
    print(f"✗ FAIL: {e}")

## 3. Query Tests (Requires Real API Key)

### 3.1 Verify API Key

In [ ]:
# Check if real API key is available
api_key = os.getenv("X_DATACORE_API_KEY")

if api_key and not api_key.startswith("test"):
    print(f"✓ Real API key found")
    print(f"  Key: {api_key[:20]}...")
    can_query = True
else:
    print(f"⚠ No real API key found")
    print(f"  Set X_DATACORE_API_KEY to run query tests")
    can_query = False

### 3.2 Test: Raw Search API

In [ ]:
# Test: Raw search API call
if can_query:
    try:
        client = Datacore()
        print("Calling search() with limit=3...")
        
        response = client.search(
            dataset_code="dataset_historical_price",
            limit=3
        )
        
        if "data" in response:
            print(f"✓ PASS: Raw search call")
            print(f"  Response keys: {list(response.get('data', {}).keys())}")
            
            if "dataDetail" in response.get("data", {}):
                records = response["data"]["dataDetail"]
                print(f"  Records returned: {len(records)}")
        else:
            print(f"✗ FAIL: Unexpected response: {list(response.keys())}")
    except Exception as e:
        print(f"✗ FAIL: {type(e).__name__}: {str(e)[:60]}")
else:
    print("⚠ Skipped: No real API key")

### 3.3 Test: Get DataFrame

In [ ]:
# Test: Get DataFrame from query
if can_query:
    try:
        client = Datacore()
        print("Calling get_dataframe() with limit=5...")
        
        df = client.get_dataframe(
            dataset_code="dataset_historical_price",
            limit=5
        )
        
        if isinstance(df, pd.DataFrame):
            print(f"✓ PASS: DataFrame query")
            print(f"  Shape: {df.shape}")
            print(f"  Columns: {list(df.columns)[:5]}")
            print()
            print("First 3 rows:")
            display(df.head(3))
        else:
            print(f"✗ FAIL: Not a DataFrame: {type(df)}")
    except Exception as e:
        print(f"✗ FAIL: {type(e).__name__}: {str(e)[:60]}")
else:
    print("⚠ Skipped: No real API key")

### 3.4 Test: Convenience Methods

In [ ]:
# Test: Convenience methods
if can_query:
    try:
        client = Datacore()
        
        methods = [
            ("get_historical_price", 5),
            ("get_fundamentals", 3),
            ("get_financial_statements", 2),
        ]
        
        print("Testing convenience methods:\n")
        
        for method_name, limit in methods:
            try:
                method = getattr(client, method_name)
                df = method(limit=limit)
                print(f"✓ {method_name:<30} → {len(df)} records")
            except Exception as e:
                print(f"✗ {method_name:<30} → Error: {type(e).__name__}")
    except Exception as e:
        print(f"✗ FAIL: {e}")
else:
    print("⚠ Skipped: No real API key")

## 4. Pagination Tests

### 4.1 Test: Pagination Generator

In [ ]:
# Test: Pagination generator
if can_query:
    try:
        client = Datacore()
        print("Testing pagination with max_pages=3, limit=5...\n")
        
        page_count = 0
        total_records = 0
        
        for df in client.paginate(
            "dataset_historical_price",
            max_pages=3,
            limit=5
        ):
            page_count += 1
            records = len(df)
            total_records += records
            print(f"  Page {page_count}: {records} records")
        
        print()
        print(f"✓ PASS: Pagination")
        print(f"  Total pages: {page_count}")
        print(f"  Total records: {total_records}")
    except Exception as e:
        print(f"✗ FAIL: {type(e).__name__}: {str(e)[:60]}")
else:
    print("⚠ Skipped: No real API key")

### 4.2 Test: Fetch All

In [ ]:
# Test: Fetch all data
if can_query:
    try:
        client = Datacore()
        print("Testing fetch_all with max_records=10, limit=5...\n")
        
        df = client.fetch_all(
            "dataset_historical_price",
            max_records=10,
            limit=5
        )
        
        print(f"✓ PASS: Fetch all")
        print(f"  Total records: {len(df)}")
        print(f"  Shape: {df.shape}")
        print()
        print("Sample data:")
        display(df.head())
    except Exception as e:
        print(f"✗ FAIL: {type(e).__name__}: {str(e)[:60]}")
else:
    print("⚠ Skipped: No real API key")

## 5. Query with Parameters

### 5.1 Test: Different Limits

In [ ]:
# Test: Different limit values
if can_query:
    try:
        client = Datacore()
        limits = [1, 5, 10]
        
        print(f"Testing different limit values:\n")
        
        for limit in limits:
            df = client.get_historical_price(limit=limit)
            actual = len(df)
            match = "✓" if actual <= limit else "⚠"
            print(f"  {match} limit={limit:<3} → got {actual} records")
        
        print()
        print(f"✓ PASS: Different limits test")
    except Exception as e:
        print(f"✗ FAIL: {type(e).__name__}: {str(e)[:60]}")
else:
    print("⚠ Skipped: No real API key")

### 5.2 Test: Query with Conditions

In [ ]:
# Test: Query with conditions
if can_query:
    try:
        client = Datacore()
        
        # Empty conditions (should work)
        conditions = []
        df = client.get_historical_price(
            conditions=conditions,
            limit=5
        )
        
        print(f"✓ PASS: Query with conditions")
        print(f"  Records fetched: {len(df)}")
        
        # Note: To use real conditions, you need to know the field names
        print()
        print("Available fields in response:")
        print(f"  {list(df.columns)}")
    except Exception as e:
        print(f"✗ FAIL: {type(e).__name__}: {str(e)[:60]}")
else:
    print("⚠ Skipped: No real API key")

## 6. Best Practices & Tips

### 6.1 Client Initialization

In [ ]:
print("""Best Practices for Client Initialization

1. ✓ GOOD: Reuse client instance
   client = Datacore()
   df1 = client.get_historical_price(limit=10)
   df2 = client.get_fundamentals(limit=10)

2. ✓ GOOD: Environment variable (recommended)
   $env:X_DATACORE_API_KEY = 'your-key'
   client = Datacore()

3. ✓ GOOD: Direct key with timeout
   client = Datacore(api_key='your-key', timeout=60)

4. ✗ BAD: Creating new client each time
   df1 = Datacore(api_key='key').get_historical_price(limit=10)
   df2 = Datacore(api_key='key').get_fundamentals(limit=10)
""")

### 6.2 Query Patterns

In [ ]:
print("""Recommended Query Patterns

1. SMALL DATASET (< 100 records)
   df = client.get_historical_price(limit=100)

2. MEDIUM DATASET (100 - 10,000 records)
   df = client.fetch_all('dataset_historical_price', max_records=5000, limit=500)

3. LARGE DATASET (> 10,000 records)
   for df in client.paginate('dataset_historical_price', limit=500):
       process(df)  # Process page by page

4. WITH FILTERING
   conditions = [{'field': 'symbol', 'value': 'ABC'}]
   df = client.get_historical_price(conditions=conditions, limit=100)

5. SPECIFIC FIELDS ONLY
   fields = ['symbol', 'price', 'date']
   df = client.get_historical_price(select_fields=fields, limit=100)
""")

### 6.3 Error Handling

In [ ]:
print("""Error Handling Patterns

1. MISSING API KEY
   try:
       client = Datacore()
   except ValueError as e:
       print('Set X_DATACORE_API_KEY environment variable')

2. API REQUEST ERROR
   try:
       df = client.get_historical_price(limit=100)
   except requests.RequestException as e:
       print('API request failed (will retry automatically)')

3. INVALID RESPONSE
   try:
       df = client.get_dataframe('dataset_invalid', limit=10)
   except ValueError as e:
       print('Response format invalid')

4. COMPLETE ERROR HANDLING
   from datacore import Datacore
   import requests
   
   try:
       client = Datacore()
       df = client.get_historical_price(limit=100)
   except ValueError as e:
       print(f'Config error: {e}')
   except requests.RequestException as e:
       print(f'API error: {e}')
   except Exception as e:
       print(f'Unexpected error: {e}')
""")

### 6.4 Performance Tips

In [ ]:
print("""Performance Tips

✓ Reuse client instance (create once, use many times)
  - Maintains persistent HTTP session
  - Faster subsequent requests

✓ Use pagination for large datasets
  - Process data page by page
  - Reduces memory usage
  - Use: client.paginate() or client.fetch_all()

✓ Set appropriate limit per page (100-500 recommended)
  - Too small: many requests
  - Too large: high memory usage

✓ Use conditions to filter on server side
  - Reduces data transfer
  - Faster than client-side filtering

✓ Select only needed fields with select_fields
  - Reduces data transfer
  - Faster parsing

✓ Automatic retry with exponential backoff
  - Retries up to 3 times on failure
  - 1s, 2s, 4s delays between retries

✓ Custom timeout for slow networks
  - Default: 30 seconds
  - Usage: Datacore(timeout=60)
""")

## Summary

In [ ]:
print("""SUMMARY

✓ Authentication:
  - Direct key: Datacore(api_key="...")
  - Environment: Set X_DATACORE_API_KEY
  - Custom timeout: Datacore(timeout=60)

✓ Datasets:
  - List: client.list_datasets()
  - Info: client.get_dataset_info("key")
  - 5+ pre-built convenience methods

✓ Queries:
  - Simple: client.get_historical_price(limit=100)
  - Raw: client.search(dataset_code="...", limit=10)
  - Generic: client.get_dataframe(dataset_code="...", limit=10)

✓ Pagination:
  - Generator: client.paginate(dataset_code, max_pages=10)
  - Fetch all: client.fetch_all(dataset_code, max_records=1000)

✓ Features:
  - Automatic retry on error (3x with backoff)
  - Session reuse (persistent HTTP connection)
  - Configurable timeout
  - Easy to extend with new datasets

✓ Documentation:
  - README.md - Quick start
  - USAGE_GUIDE.md - Comprehensive usage
  - EXTENSION_GUIDE.md - Adding new datasets
""")